# 라이브러리 설치

In [1]:
%pip install torch torchvision scikit-learn pandas onnx onnxscript

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [11]:
from torchvision import transforms,datasets
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss
from torch.utils.data import DataLoader
import torch
import os
import numpy as np
import cv2
from model import Model
from torch.nn import CrossEntropyLoss
import copy
from torch.optim import Adam
import pandas as pd
from tqdm import tqdm

In [12]:
BATCH=32
EPOCHS=300
LR=0.001
IMGZ = (384,384)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

In [13]:
img_dir = './dataset/'
folderNames = ['train', 'valid']
final_result = {}

for folderName in folderNames:
    imgs = []
    current_path = os.path.join(img_dir, folderName)
    
    if not os.path.exists(current_path):
        continue
    
    # 클래스 폴더 순회 (예: spring, summer...)
    for className in os.listdir(current_path):
        classPath = os.path.join(current_path, className)
        if not os.path.isdir(classPath): continue
        
        # 이미지 파일 순회
        for img_name in os.listdir(classPath):
            fileName = os.path.join(classPath, img_name)
            
            img = cv2.imread(fileName)
            if img is None: continue
            
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, IMGZ)
            img = img / 255.0
            imgs.append(img)
            
    # 핵심: 모든 클래스의 이미지를 다 모은 후, 루프 밖에서 단 한 번 변환
    if imgs:
        imgs_array = np.array(imgs)
        m = np.mean(imgs_array, axis=(0, 1, 2))
        s = np.std(imgs_array, axis=(0, 1, 2))
        final_result[folderName] = {"mean": m, "std": s}
    else:
        final_result[folderName] = {"mean": "No Data", "std": "No Data"}

# 최종 출력
for key, value in final_result.items():
    print(f"[{key.upper()}] Mean: {value['mean']}")
    print(f"[{key.upper()}] Std: {value['std']}")

[TRAIN] Mean: [0.62226292 0.51213687 0.48001183]
[TRAIN] Std: [0.29438873 0.26803289 0.26197309]
[VALID] Mean: [0.68602493 0.54892893 0.4978932 ]
[VALID] Std: [0.25627352 0.2370695  0.23574143]


In [ ]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize(IMGZ), # 이미지 크기 조정 (Resize images)
        transforms.RandomRotation(15),             # 최대 15도 회전
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)), # 미세 평행 이동
        transforms.ColorJitter(brightness=0.2, contrast=0.2), # 밝기, 대비만 조절 (Hue 금지)
        transforms.RandomHorizontalFlip(), # 데이터 증강: 좌우 반전 (Data augmentation: Horizontal flip)
        transforms.ToTensor(), # 이미지를 텐서로 변환 (Convert image to tensor)
        # 정규화: ImageNet 평균과 표준편차 사용 (Normalize: Use ImageNet mean and std dev)
        transforms.Normalize([0.62226292, 0.51213687, 0.48001183],
        [0.29438873, 0.26803289, 0.26197309])
    ]),
    'valid': transforms.Compose([
        transforms.Resize(IMGZ), # 이미지 크기 조정 (Resize images)
        transforms.RandomRotation(15),             # 최대 15도 회전
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)), # 미세 평행 이동
        transforms.ColorJitter(brightness=0.2, contrast=0.2), # 밝기, 대비만 조절 (Hue 금지)
        transforms.ToTensor(), # 이미지를 텐서로 변환 (Convert image to tensor)
        # 정규화 (Normalize)
        transforms.Normalize( [0.68602493, 0.54892893, 0.4978932 ],
        [0.25627352, 0.2370695,  0.23574143])
    ]),
}

data_dir = './dataset' # 데이터셋 디렉토리 설정 (Set dataset directory)
# 데이터셋 불러오기 (Load datasets)
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
                  for x in ['train', 'valid']}
# 데이터 로더 설정 (Set up data loaders)
dataloaders = {x: DataLoader(image_datasets[x], batch_size=BATCH, shuffle=True)
              for x in ['train', 'valid']}

In [15]:
model=Model(len(image_datasets['train'].classes))

In [16]:
optimizer=Adam(model.head.parameters(),lr=LR)
criterion = CrossEntropyLoss()

In [17]:
def SaveDataFrame(result,dir):
    df=pd.DataFrame(result)
    df.index=df.index+1
    df.to_csv(s.path.join(dir, "results.csv"))

In [18]:
def get_save_dir(base_name="train"):
    if not os.path.exists("run"):
        os.mkdir("run")
    base_name=os.path.join("run",base_name)
    if not os.path.exists(base_name):
        os.mkdir(base_name)
        return base_name
    
    i = 1
    while True:
        new_dir = f"{base_name}{i}"
        if not os.path.exists(new_dir):
            os.mkdir(new_dir)
            return new_dir
        i += 1


In [ ]:
def calculate_metrics(labels, preds):
    return {
        "acc": accuracy_score(labels, preds),
        "prec": precision_score(labels, preds, average='weighted', zero_division=0),
        "rec": recall_score(labels, preds, average='weighted', zero_division=0),
        "f1": f1_score(labels, preds, average='weighted', zero_division=0)
    }
result={
    "train_accuracy_score":[],
    "train_loss":[],
    "train_precision_score":[],
    "train_recall_score":[],
    "train_f1_score":[],

    "valid_accuracy_score":[],
    "valid_loss":[],
    "valid_precision_score":[],
    "valid_recall_score":[],
    "valid_f1_score":[]
}
# 2. 결과 저장 함수
def log_results(result, phase, loss, metrics, save_dir):
    result[f"{phase}_loss"].append(loss)
    result[f"{phase}_accuracy_score"].append(float(metrics['acc']))
    result[f"{phase}_precision_score"].append(metrics['prec'])
    result[f"{phase}_recall_score"].append(metrics['rec'])
    result[f"{phase}_f1_score"].append(metrics['f1'])
    if phase=='valid':
        # CSV 저장 (Ultralytics 스타일)
        df = pd.DataFrame(result)
        df.index = df.index + 1
        df.to_csv(os.path.join(save_dir, "results.csv"), index_label="epoch")

# --- 메인 학습 루프 ---
save_dir = get_save_dir()
os.makedirs(save_dir, exist_ok=True)
best_acc = 0.0
best_loss = float('inf')

for epoch in range(EPOCHS):

    for phase in ['train', 'valid']:
        desc = f'{phase.upper()} | Ep {epoch+1}/{EPOCHS} | Img {IMGZ[0]} | {DEVICE}'
        pbar = tqdm(dataloaders[phase], desc=desc, unit='batch', leave=True)
        model.train() if phase == 'train' else model.eval()
        
        running_loss, all_preds, all_labels = 0.0, [], []

        for inputs, labels in dataloaders[phase]:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()
            pbar.set_postfix(loss=f"{loss.item():.4f}")
            running_loss += loss.item() * inputs.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

        # 지표 산출
        epoch_loss = running_loss / len(image_datasets[phase])
        metrics = calculate_metrics(all_labels, all_preds)

        # BEST 모델 갱신 (검증 단계 기준)
        if phase == "valid":
            is_better = (metrics['acc'] > best_acc) or (metrics['acc'] == best_acc and epoch_loss < best_loss)
            if is_better:
                best_acc, best_loss = metrics['acc'], epoch_loss
                torch.save(model.state_dict(), os.path.join(save_dir, "best.pt"))
                print(f"⭐ New Best: Acc {best_acc:.4f} Loss {best_loss:.4f}")

        # 결과 기록 및 로그 출력
        log_results(result, phase, epoch_loss, metrics, save_dir)
        torch.save(model.state_dict(), os.path.join(save_dir, "last.pt"))
        print(f'{phase.upper()} | Loss: {epoch_loss:.4f} Acc: {metrics["acc"]:.4f} F1: {metrics["f1"]:.4f}')

print("학습 완료.")


TRAIN | Ep 1/300 | Img 384 | cpu:   0%|          | 0/59 [02:52<?, ?batch/s, loss=1.0125]







In [ ]:
dummy_input = torch.randn(1, 3, IMGZ[0],IMGZ[1], device=DEVICE)
onnx_file_path = "model.onnx"
torch.onnx.export(
    model,                  # 이제 '가중치'가 아니라 '모델 객체'를 넘깁니다.
    dummy_input,            
    onnx_file_path,         
    export_params=True,     
    opset_version=12,       
    do_constant_folding=True,
    input_names=['input'],   
    output_names=['output']
)